# SecureOps Assistant — RAG Implementation
**AAI Tech Talks Hackathon 2026 · MSc Applied AI · WMG, University of Warwick**


## Step 0 — Install dependencies

In [1]:
%pip install -q chromadb sentence-transformers pypdf google-genai requests crawl4ai

# ── SSL fix for Windows ───────────────────────────────────────────────────────
# One malformed cert in the Windows cert store makes ssl.create_default_context()
# fail with "[ASN1: NOT_ENOUGH_DATA]". aiohttp (a crawl4ai dependency) calls that
# at import time, so crawl4ai can't even be imported. CPython loads the whole
# store as a single blob, so one bad cert poisons all of them.
#
# Patch the loader to validate certs ONE AT A TIME and skip the bad one. This
# MUST run before crawl4ai / aiohttp is first imported in the kernel.
import ssl, _ssl

def _safe_load_windows_store_certs(self, storename, purpose):
    certs = bytearray()
    try:
        for cert, encoding, trust in _ssl.enum_certificates(storename):
            if encoding == "x509_asn" and (trust is True or purpose.oid in trust):
                try:
                    self.load_verify_locations(cadata=cert)  # add good certs individually
                    certs.extend(cert)
                except ssl.SSLError:
                    continue                                  # skip the malformed cert
    except PermissionError:
        pass
    return certs

ssl.SSLContext._load_windows_store_certs = _safe_load_windows_store_certs

# crawl4ai-setup just installs the Playwright browser; call playwright directly
# so it runs in this kernel (where the patch above is active) instead of a subprocess.
!playwright install chromium

print("✅ Dependencies installed (Windows cert store patched)")

Note: you may need to restart the kernel to use updated packages.
✅ Dependencies installed (Windows cert store patched)


## Step 1 — Download NIST PDFs

Downloads NIST SP 800-82 Rev. 3 and NIST CSF 2.0 directly from NIST's public servers.
Both are public domain US government documents — no login or authentication required.
If already downloaded, the cell skips them.

In [2]:
import requests, pathlib

CORPUS_DIR     = pathlib.Path("corpus")
ADVISORIES_DIR = CORPUS_DIR / "advisories"
CORPUS_DIR.mkdir(exist_ok=True)
ADVISORIES_DIR.mkdir(exist_ok=True)

HEADERS = {"User-Agent": "Mozilla/5.0 (SecureOps-Hackathon student notebook)"}

PDFS = {
    "nist_sp800_82r3.pdf": "https://nvlpubs.nist.gov/nistpubs/SpecialPublications/NIST.SP.800-82r3.pdf",
    "nist_csf_2_0.pdf":    "https://nvlpubs.nist.gov/nistpubs/CSWP/NIST.CSWP.29.pdf",
}

for fname, url in PDFS.items():
    dest = CORPUS_DIR / fname
    if dest.exists():
        print(f"✅ already downloaded: {fname}")
        continue
    print(f"⬇️  downloading {fname} ...")
    resp = requests.get(url, headers=HEADERS, timeout=120)
    resp.raise_for_status()
    dest.write_bytes(resp.content)
    print(f"✅ saved {fname} ({len(resp.content)/1e6:.1f} MB)")

✅ already downloaded: nist_sp800_82r3.pdf
✅ already downloaded: nist_csf_2_0.pdf


## Step 2 — Fetch CISA ICS Advisories

Uses `crawl4ai` to paginate through `cisa.gov/news-events/ics-advisories?page=N`.

Each advisory is saved as an individual `.md` file with **YAML frontmatter** — structured
metadata fields (vendor, CVSS score, CVEs, sectors, release date) in the `---` block at the
top, followed by the full markdown body (sections, tables, bullet lists preserved by crawl4ai).

This format enables:
- **Metadata filtering** at query time (`vendor`, `cvss_score`, `release_date`, `sectors`)
- **Structure-aware chunking** — `##` section boundaries are preserved in the body
- **Human readability** — every advisory is inspectable as a plain text file

Files saved to `corpus/advisories/ICSA-XX-XXX-XX.md`, one per advisory.

In [3]:
import asyncio, re, sys, threading
from crawl4ai import AsyncWebCrawler

BASE_URL     = "https://www.cisa.gov"
LISTING_URL  = "https://www.cisa.gov/news-events/ics-advisories?page={page}"
N_ADVISORIES = 100

# ── Windows event-loop helper ─────────────────────────────────────────────────
# Playwright spawns its browser driver as a subprocess. On Windows, asyncio
# subprocesses only work on a ProactorEventLoop, but Jupyter runs on a
# SelectorEventLoop (which raises NotImplementedError for subprocesses). We can't
# swap a running loop, so run the crawl coroutine in a dedicated thread that owns
# a fresh ProactorEventLoop.
def run_async(coro_factory, *args, **kwargs):
    box = {}
    def runner():
        loop = asyncio.ProactorEventLoop() if sys.platform == "win32" else asyncio.new_event_loop()
        asyncio.set_event_loop(loop)
        try:
            box["result"] = loop.run_until_complete(coro_factory(*args, **kwargs))
        except BaseException as exc:
            box["error"] = exc
        finally:
            loop.close()
    t = threading.Thread(target=runner)
    t.start()
    t.join()
    if "error" in box:
        raise box["error"]
    return box.get("result")

# ── Helpers ───────────────────────────────────────────────────────────────────

def clean_markdown(markdown):
    """Strip .gov header banner — start content from the first H1 heading."""
    lines = markdown.split("\n")
    for i, line in enumerate(lines):
        if line.startswith("# "):
            return "\n".join(lines[i:])
    return markdown

def extract_metadata(text, url):
    """Parse YAML frontmatter fields from crawl4ai markdown."""

    alert_code = url.rstrip("/").split("/")[-1].upper()

    title_match = re.search(r'^#\s+(.+)', text, re.MULTILINE)
    title = title_match.group(1).strip() if title_match else alert_code

    date_match = re.search(r'Release Date\s*\n([^\n]+)', text)
    release_date_raw = date_match.group(1).strip() if date_match else ""
    try:
        from datetime import datetime
        release_date = datetime.strptime(release_date_raw, "%B %d, %Y").strftime("%Y-%m-%d")
    except Exception:
        release_date = release_date_raw

    cvss_match   = re.search(r'v(\d)\s+(\d+\.\d+)', text)
    cvss_version = f"v{cvss_match.group(1)}" if cvss_match else ""
    cvss_score   = float(cvss_match.group(2)) if cvss_match else None

    vendor = ""
    for pattern in [
        r'\*\*Vendor\*\*\s+([^|*\n]+)',
        r'\*\*Vendor:\*\*\s*\n\s*([^\n]+)',
    ]:
        m = re.search(pattern, text)
        if m:
            vendor = m.group(1).strip()
            break

    cves    = sorted(set(re.findall(r'CVE-\d{4}-\d+', text)))
    cwes    = sorted(set(re.findall(r'CWE-\d+', text)))

    sectors = []
    sectors_match = re.search(r'Critical Infrastructure Sectors?:?\*?\*?\s*([^\n]+)', text)
    if sectors_match:
        sectors = [s.strip() for s in sectors_match.group(1).split(",")]

    countries_match = re.search(r'Countries/Areas Deployed:?\*?\*?\s*([^\n]+)', text)
    countries = countries_match.group(1).strip() if countries_match else ""

    return {
        "alert_code":   alert_code,
        "title":        title,
        "url":          url,
        "release_date": release_date,
        "vendor":       vendor,
        "cvss_version": cvss_version,
        "cvss_score":   cvss_score,
        "cves":         cves,
        "cwe":          cwes,
        "sectors":      sectors,
        "countries":    countries,
    }

def build_frontmatter(meta):
    """Build the YAML frontmatter block as a string."""
    lines = ["---"]
    lines.append(f'alert_code: {meta["alert_code"]}')
    lines.append(f'title: "{meta["title"]}"')
    lines.append(f'url: {meta["url"]}')
    lines.append(f'release_date: "{meta["release_date"]}"')
    lines.append(f'vendor: "{meta["vendor"]}"')
    lines.append(f'cvss_version: "{meta["cvss_version"]}"')
    lines.append(f'cvss_score: {meta["cvss_score"]}')
    if meta["cves"]:
        lines.append("cves:")
        for cve in meta["cves"]:
            lines.append(f"  - {cve}")
    if meta["cwe"]:
        lines.append("cwe:")
        for cwe in meta["cwe"]:
            lines.append(f"  - {cwe}")
    if meta["sectors"]:
        lines.append("sectors:")
        for sector in meta["sectors"]:
            lines.append(f'  - "{sector}"')
    lines.append(f'countries: "{meta["countries"]}"')
    lines.append("---")
    return "\n".join(lines)

# ── Main crawler ──────────────────────────────────────────────────────────────

async def collect_all_advisories(max_advisories=N_ADVISORIES):
    saved = 0
    links = []

    async with AsyncWebCrawler() as crawler:

        page = 0
        while len(links) < max_advisories:
            result = await crawler.arun(url=LISTING_URL.format(page=page))
            if not result.success:
                print(f"  ⚠️ Failed at listing page {page}, stopping.")
                break

            internal = result.links.get("internal", [])
            new_links = [
                BASE_URL + l["href"] if l["href"].startswith("/") else l["href"]
                for l in internal
                if "/ics-advisories/icsa-" in l.get("href", "")
            ]
            new_links = [l for l in new_links if l not in links]

            if not new_links:
                print(f"  No more advisory links at page {page}, stopping.")
                break

            links.extend(new_links)
            print(f"  Page {page}: {len(new_links)} links (total: {len(links)})")
            page += 1
            await asyncio.sleep(1)

        links = links[:max_advisories]
        print(f"\nFetching {len(links)} advisory pages...")

        for i, link in enumerate(links):
            result = await crawler.arun(url=link)
            if result.success and result.markdown and len(result.markdown) > 500:
                cleaned  = clean_markdown(result.markdown)
                meta     = extract_metadata(cleaned, link)
                fm       = build_frontmatter(meta)
                md_file  = ADVISORIES_DIR / f"{meta['alert_code']}.md"
                md_file.write_text(f"{fm}\n\n{cleaned}", encoding="utf-8")
                saved += 1
                print(f"  ✅ [{i+1}/{len(links)}] {meta['alert_code']} — {meta['title'][:55]}")
            else:
                print(f"  ⚠️ skipped {link}")
            await asyncio.sleep(1)

    return saved

# Run on a ProactorEventLoop in a worker thread (see run_async note above).
saved = run_async(collect_all_advisories, N_ADVISORIES)

print(f"\n✅ corpus ready: 2 NIST PDFs + {saved} CISA advisories")
print(f"   advisories saved to: {ADVISORIES_DIR}/")

[INIT].... → Crawl4AI 0.9.0 

[FETCH]... ↓ https://www.cisa.gov/news-events/ics-advisories?page=0                                               | ✓ | ⏱: 1.02s 

[SCRAPE].. ◆ https://www.cisa.gov/news-events/ics-advisories?page=0                                               | ✓ | ⏱: 0.29s 

[COMPLETE] ● https://www.cisa.gov/news-events/ics-advisories?page=0                                               | ✓ | ⏱: 1.34s 

  Page 0: 9 links (total: 9)


...

  ✅ [99/100] ICSA-26-111-03 — Siemens SINEC NMS


[FETCH]... ↓ https://www.cisa.gov/news-events/ics-advisories/icsa-26-111-02                                       | ✓ | ⏱: 0.59s 

[SCRAPE].. ◆ https://www.cisa.gov/news-events/ics-advisories/icsa-26-111-02                                       | ✓ | ⏱: 0.06s 

[COMPLETE] ● https://www.cisa.gov/news-events/ics-advisories/icsa-26-111-02                                       | ✓ | ⏱: 0.66s 

  ✅ [100/100] ICSA-26-111-02 — Siemens RUGGEDCOM CROSSBOW Secure Access Manager Primar

✅ corpus ready: 2 NIST PDFs + 100 CISA advisories
   advisories saved to: corpus\advisories/


## Step 3 — Ingest MITRE ATT&CK for ICS

Downloads the official STIX 2.1 bundle from the MITRE GitHub repository — one file,
no scraping needed. The bundle contains all ICS techniques, tactics, mitigations, and
the relationships linking them.

Each technique is saved as an individual `.md` file with **YAML frontmatter** containing:
`technique_id`, `name`, `tactics`, `tactic_ids`, `platforms`, `is_subtechnique`.

The markdown body contains the full description and all linked mitigations as subsections.
This directly answers the brief's ATT&CK question:
*"Which ATT&CK for ICS techniques involve manipulation of control logic?"*

Files saved to `corpus/attck/T0855.md`, one per technique.

In [8]:
import requests, json, pathlib

CORPUS_DIR = pathlib.Path("corpus")
ATTCK_DIR  = CORPUS_DIR / "attck"
ATTCK_DIR.mkdir(exist_ok=True)

STIX_URL = "https://raw.githubusercontent.com/mitre-attack/attack-stix-data/master/ics-attack/ics-attack.json"

# ── Download ──────────────────────────────────────────────────────────────────

print("⬇️  Downloading MITRE ATT&CK for ICS STIX bundle...")
resp = requests.get(STIX_URL, timeout=120)
resp.raise_for_status()
objects = resp.json().get("objects", [])
print(f"✅ Downloaded — {len(objects)} STIX objects")

# ── Build lookups ─────────────────────────────────────────────────────────────

# Tactic: shortname (e.g. 'inhibit-response-function') → {name, tactic_id}
tactics = {}
for obj in objects:
    if obj.get("type") == "x-mitre-tactic":
        shortname = obj.get("x_mitre_shortname", "")
        tactic_id = next(
            (r["external_id"] for r in obj.get("external_references", [])
             if r.get("source_name") == "mitre-attack"), ""
        )
        tactics[shortname] = {"name": obj.get("name", ""), "tactic_id": tactic_id}

# Mitigations: STIX id → {name, description}
mitigations = {
    obj["id"]: {"name": obj.get("name", ""), "description": obj.get("description", "")}
    for obj in objects
    if obj.get("type") == "course-of-action"
    and not obj.get("x_mitre_deprecated")
    and not obj.get("revoked")
}

# Technique → list of mitigations, via 'mitigates' relationships
technique_mitigations = {}
for obj in objects:
    if obj.get("type") == "relationship" and obj.get("relationship_type") == "mitigates":
        src, tgt = obj.get("source_ref", ""), obj.get("target_ref", "")
        if src in mitigations:
            technique_mitigations.setdefault(tgt, []).append(mitigations[src])

print(f"   Tactics loaded     : {len(tactics)}")
print(f"   Mitigations loaded : {len(mitigations)}")

# ── Parse & save techniques ───────────────────────────────────────────────────

saved = 0
for obj in objects:
    if obj.get("type") != "attack-pattern":
        continue
    if obj.get("revoked") or obj.get("x_mitre_deprecated"):
        continue

    # Technique ID and URL from external references
    technique_id, technique_url = "", ""
    for ref in obj.get("external_references", []):
        if ref.get("source_name") == "mitre-attack":
            technique_id  = ref.get("external_id", "")
            technique_url = ref.get("url", "")
    if not technique_id:
        continue

    name             = obj.get("name", "")
    description      = obj.get("description", "")
    is_subtechnique  = obj.get("x_mitre_is_subtechnique", False)
    platforms        = obj.get("x_mitre_platforms", [])
    parent_id        = technique_id.split(".")[0] if is_subtechnique else ""

    # Tactics from kill_chain_phases
    tactic_names, tactic_ids = [], []
    for phase in obj.get("kill_chain_phases", []):
        shortname = phase.get("phase_name", "")
        if shortname in tactics:
            tactic_names.append(tactics[shortname]["name"])
            tactic_ids.append(tactics[shortname]["tactic_id"])

    related_mitigations = technique_mitigations.get(obj["id"], [])

    # ── YAML frontmatter ──────────────────────────────────────────────────────
    fm = ["---"]
    fm.append(f"technique_id: {technique_id}")
    fm.append(f'name: "{name}"')
    if tactic_names:
        fm.append("tactics:")
        for t in tactic_names:
            fm.append(f'  - "{t}"')
    if tactic_ids:
        fm.append("tactic_ids:")
        for t in tactic_ids:
            fm.append(f"  - {t}")
    if platforms:
        fm.append("platforms:")
        for p in platforms:
            fm.append(f'  - "{p}"')
    fm.append(f"is_subtechnique: {str(is_subtechnique).lower()}")
    if parent_id:
        fm.append(f"parent_technique: {parent_id}")
    fm.append("source: MITRE ATT&CK for ICS")
    fm.append(f"url: {technique_url}")
    fm.append("---")
    frontmatter = "\n".join(fm)

    # ── Markdown body ─────────────────────────────────────────────────────────
    body = [f"# {name}", "", "## Description", description]

    if related_mitigations:
        body += ["", "## Mitigations"]
        for m in related_mitigations:
            body += [f"\n### {m['name']}", m["description"]]

    # ── Save file ─────────────────────────────────────────────────────────────
    filename  = technique_id.replace(".", "-")   # T0855.001 → T0855-001
    filepath  = ATTCK_DIR / f"{filename}.md"
    body_text = "\n".join(body)                   # precompute — f-strings can't hold a backslash (Py<3.12)
    filepath.write_text(f"{frontmatter}\n\n{body_text}", encoding="utf-8")
    saved += 1
    print(f"  ✅ {technique_id} — {name}")

print(f"\n✅ {saved} ATT&CK for ICS techniques saved to {ATTCK_DIR}/")

⬇️  Downloading MITRE ATT&CK for ICS STIX bundle...
✅ Downloaded — 2174 STIX objects
   Tactics loaded     : 12
   Mitigations loaded : 52
  ✅ T0881 — Service Stop
  ✅ T0836 — Modify Parameter
  ✅ T0821 — Modify Controller Tasking
  ✅ T0887 — Wireless Sniffing
  ✅ T0829 — Loss of View
  ✅ T1691.001 — Command Message
  ✅ T0800 — Activate Firmware Update Mode
  ✅ T0831 — Manipulation of Control
  ✅ T0814 — Denial of Service
  ✅ T0894 — System Binary Proxy Execution
  ✅ T0807 — Command-Line Interface
  ✅ T0861 — Point & Tag Identification
  ✅ T0816 — Device Restart/Shutdown
  ✅ T0863 — User Execution
  ✅ T0860 — Wireless Compromise
  ✅ T0858 — Change Operating Mode
  ✅ T0878 — Alarm Suppression
  ✅ T0868 — Detect Operating Mode
  ✅ T0837 — Loss of Protection
  ✅ T0801 — Monitor Process State
  ✅ T0853 — Scripting
  ✅ T0888 — Remote System Information Discovery
  ✅ T0845 — Program Upload
  ✅ T0819 — Exploit Public-Facing Application
  ✅ T1691 — Block Operational Technology Message
  ✅ T081

## Step 4 — Clean & normalize the corpus → `new_corpus/`

Steps 1–3 capture the **raw "bronze" layer** in `corpus/` — faithful to the source, but
noisy. This step produces the **cleaned "silver" layer** in `new_corpus/`, which is what the
ingestion / chunking pipeline reads from. Separating raw capture from cleaning keeps the
pipeline reproducible and the raw data re-inspectable.

**Why this matters for retrieval:** roughly **40% of every CISA advisory page is boilerplate**
— legal notices, revision-history tables, "Related Advisories", and the cisa.gov site footer
(social links, FOIA/USA.gov nav, empty tables). The *Recommended Practices* paragraph is also
near-identical across all 100 advisories. Embedded as-is, that junk creates ~100 near-duplicate
chunks that pollute similarity search and waste the embedder's 512-token budget.

**What the cleaner does (content is preserved, only non-content is removed):**

| Source | Cleaning |
|---|---|
| CISA advisories | Truncate at the first boilerplate marker (Legal Notice / Revision History / Tags / Related Advisories / footer); keep Summary, Background, Vulnerabilities + per-CVE blocks, Affected Products, Metrics, Acknowledgments, General Recommendations. Strip `(link is external)` and nav links. |
| MITRE ATT&CK | Remove inline `(Citation: …)` markers; unwrap `[label](url)` cross-links to their labels. |
| NIST PDFs | Copied through unchanged — PDFs are binary and are cleaned at **parse** time (running header/footer, page-number offset) in the chunking stage, not here. |

> **Note:** this cell reads the already-downloaded `corpus/` and rewrites it into `new_corpus/` —
> it does **not** re-scrape, so it runs in seconds and is fully deterministic. YAML frontmatter
> (`vendor`, `cvss_score`, `cves`, `release_date`, …) is carried over verbatim for metadata filtering.

In [ ]:
import re, shutil, pathlib

# ── Layers ────────────────────────────────────────────────────────────────────
# corpus/      = "bronze": raw capture, faithful to the source (Steps 1-3)
# new_corpus/  = "silver": same docs + same YAML frontmatter, boilerplate removed
RAW_DIR = pathlib.Path("corpus")
NEW_DIR = pathlib.Path("new_corpus")
(NEW_DIR / "advisories").mkdir(parents=True, exist_ok=True)
(NEW_DIR / "attck").mkdir(parents=True, exist_ok=True)

# ── Shared helpers ────────────────────────────────────────────────────────────

def split_frontmatter(text):
    """Return (frontmatter_block_with_delimiters, body). YAML is preserved verbatim."""
    if text.startswith("---"):
        end = text.find("\n---", 3)
        if end != -1:
            line_end = text.index("\n", end + 1)   # end of the closing '---' line
            return text[:line_end + 1], text[line_end + 1:]
    return "", text

LINK_RE       = re.compile(r"\[([^\]]+)\]\([^)]*\)")    # [label](url) -> label
EMPTY_LINK_RE = re.compile(r"\[\s*\]\([^)]*\)")          # [](url)      -> (removed)
EXTLINK_RE    = re.compile(r"\s*\(link is external\)")
ANGLE_URL_RE  = re.compile(r"<https?://[^>]+>")
HR_RE         = re.compile(r"^\s*\*\s\*\s\*\s*$", re.MULTILINE)
MULTINL_RE    = re.compile(r"\n{3,}")

def strip_links(text):
    """Drop URL noise: keep human-readable link labels, remove the URLs themselves.
    (Downstream citations are by doc/section, not URL — the source URL stays in the
    frontmatter, so embedding raw URLs here would only add retrieval noise.)"""
    text = EXTLINK_RE.sub("", text)
    text = ANGLE_URL_RE.sub("", text)
    text = EMPTY_LINK_RE.sub("", text)
    text = LINK_RE.sub(r"\1", text)
    return text

def tidy(text):
    text = HR_RE.sub("", text)                  # drop '* * *' horizontal rules
    text = MULTINL_RE.sub("\n\n", text)         # collapse runs of blank lines
    return text.strip() + "\n"

# ── CISA advisory cleaner ─────────────────────────────────────────────────────
# Cut the body at the EARLIEST non-content marker. Everything above (Summary,
# Background, Vulnerabilities + per-CVE blocks, Affected Products, Metrics,
# Acknowledgments, General Recommendations) is kept; the legal/revision/tags/
# "Related Advisories" tail and the cisa.gov site footer/nav are discarded.
ADVISORY_CUT_MARKERS = [
    r"^##\s+Legal Notice", r"^##\s+Terms of Use", r"^##\s+Additional Resources",
    r"^##\s+Advisory Conversion Disclaimer", r"^##\s+Revision History",
    r"^##\s+Please share your thoughts", r"^##\s+Related Advisories",
    r"^###?\s+Tags", r"^\s*\[Return to top\]",
]
CUT_RE            = re.compile("|".join(ADVISORY_CUT_MARKERS), re.MULTILINE | re.IGNORECASE)
NAV_NOISE_RE      = re.compile(r"^\s*\[(?:Expand All|View CSAF|View CVE Details|\*\*View CSAF).*$", re.MULTILINE)
RELATED_TOPICS_RE = re.compile(r"^Related topics:.*(?:\n.*)?$", re.MULTILINE)

def clean_advisory(text):
    fm, body = split_frontmatter(text)
    m = CUT_RE.search(body)
    if m:
        body = body[:m.start()]
    body = NAV_NOISE_RE.sub("", body)
    body = RELATED_TOPICS_RE.sub("", body)
    body = strip_links(body)
    return fm + "\n" + tidy(body)

# ── MITRE ATT&CK cleaner ──────────────────────────────────────────────────────
# Remove inline "(Citation: ...)" markers and unwrap mitigation cross-links.
CITATION_RE = re.compile(r"\s*\(Citation:[^)]*\)")

def clean_attck(text):
    fm, body = split_frontmatter(text)
    body = CITATION_RE.sub("", body)
    body = strip_links(body)
    return fm + "\n" + tidy(body)

# ── PDFs: binary — cleaned at PARSE time (header/footer/page-offset), so copy ──
for pdf in ["nist_sp800_82r3.pdf", "nist_csf_2_0.pdf"]:
    src = RAW_DIR / pdf
    if src.exists():
        shutil.copy2(src, NEW_DIR / pdf)
        print(f"  copied {pdf}")
    else:
        print(f"  missing {pdf} — run Step 1 first")

# ── Clean advisories & ATT&CK, reporting the size reduction ───────────────────
def clean_dir(subdir, cleaner, unit, label):
    raw_t = cln_t = 0
    files = sorted((RAW_DIR / subdir).glob("*.md"))
    for f in files:
        raw = f.read_text(encoding="utf-8")
        cleaned = cleaner(raw)
        (NEW_DIR / subdir / f.name).write_text(cleaned, encoding="utf-8")
        raw_t += len(raw); cln_t += len(cleaned)
    pct = 100 * (1 - cln_t / raw_t) if raw_t else 0
    print(f"  {subdir}: {len(files)} files, "
          f"{raw_t/unit:.2f}{label} -> {cln_t/unit:.2f}{label} ({pct:.0f}% smaller)")

clean_dir("advisories", clean_advisory, 1e6, "MB")
clean_dir("attck",      clean_attck,    1e3, "KB")

print(f"\n✅ silver corpus ready at: {NEW_DIR.resolve()}")
print("   This is the directory the ingestion/chunking pipeline should read from.")